Autogras & Gradientes

Para que una red neuronal aprenda necesita saber en q direccion ajsutas sus pesos para equivocarse menos, esto se hace calculando derivadas -> gradientes, pytroch tiene un moto llamado : AutoGrad q ahce estas derividadas automaticamente por muy complejo que sea


Por defecto los tensores de pytorch no rastrean las operaciones que se les aplican para ahorrar memoria, por lo tanto hay que activarlo -> requires_grad = true

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)

y = x**2

print(y) 
# imprime --> tensor(9., grad_fn=<PowBackward0>)
# 'grad_fn' significa que PyTorch ha guardado la ecuación para derivar luego.

tensor(9., grad_fn=<PowBackward0>)


funcion .backward() .grad()

una vez obtenido ese resultado, se llama backward que esto realiza el calculo de las derivadas y en cadena backproagation

In [5]:
y.backward()

# El resultado de la derivada se guarda autom. en x.grad
print(f"El gradiente (derivada) de y respecto a x es: {x.grad}")

El gradiente (derivada) de y respecto a x es: 6.0


Si $y = x^2$, la derivada es $dy/dx = 2x$.
Como dijimos que $x = 3.0$, la derivada en ese punto es $2 * 3.0 = 6.0$

Cuando estas evaluando tu modelo haciendo inferencia o calculando el error en el conjunto de validacion no quieres que la red aprenda, pq mucho gasto memeoria y tiempo entonces se apga temporalamente 

In [6]:
# Imagina que aquí estás haciendo la inferencia para tu práctica de Iris
with torch.no_grad():
    y_prediccion = x * 5  # Aquí PyTorch no rastreará nada ni guardará memoria extra
    print(y_prediccion.requires_grad) # Esto dará False

False


Ejercicios basicos practicos: 

Ejercicio 1:Crea un tensor x con el valor 2.0 que requiera cálculo de gradientes.Luego, implementa la siguiente función matemática: $z = 3x^3 + 4x$Finalmente, haz que PyTorch calcule la derivada de $z$ respecto a $x$ y muestra el valor del gradiente por pantalla.

Ejercicio 2:Escribe el código para realizar la operación $w = x + 2$ (usando el mismo x de antes), pero asegúrate de hacerlo dentro de un bloque donde el motor Autograd de PyTorch esté desactivado, simulando que estamos en fase de inferencia.

In [9]:
x = torch.tensor(2., requires_grad=True)

y = 3*x**3 + 4*x

y.backward()

print(f"El gradiente de y respecto a x es: {x.grad}")



El gradiente de y respecto a x es: 40.0


In [10]:
with torch.no_grad():
    y_prediccion = x+2
    print(y_prediccion.requires_grad) 

False


OTROS EJERCICIOS : 

In [ ]:
# -*- coding: utf-8 -*-
"""
PyTorch: Tensors and autograd
-------------------------------

A third order polynomial, trained to predict :math:`y=\sin(x)` from :math:`-\pi`
to :math:`\pi` by minimizing squared Euclidean distance.

This implementation computes the forward pass using operations on PyTorch
Tensors, and uses PyTorch autograd to compute gradients.


A PyTorch Tensor represents a node in a computational graph. If ``x`` is a
Tensor that has ``x.requires_grad=True`` then ``x.grad`` is another Tensor
holding the gradient of ``x`` with respect to some scalar value.
"""
import torch
import math

dtype = torch.float
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x = torch.linspace(-math.pi, math.pi, 2000, dtype=dtype)
y = torch.sin(x)

# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y using operations on Tensors.

    # Computa la expresión y = a + b x + c x^2 + d x^3
    # ESCRIBE TU CÓDIGO AQUÍ
    y_pred = a+b+c*(x**2)+d*(x**3)

    # Compute and print loss using operations on Tensors.
    # Now loss is a Tensor of shape (1,)
    #loss.item() gets the scalar value held in the loss.
    # Calcula la pérdida cuadrática media entre y_pred y y
    # ESCRIBE TU CÓDIGO AQUÍ

    loss = ((y_pred - y) ** 2).sum()

    if t % 100 == 99:
        print(t, loss.item())

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be Tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():
        a -= learning_rate * a.grad
        # Recalcula b, c y d de forma similar a a
        # ESCRIBE TU CÓDIGO AQUÍ
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad      
        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

# Solución: https://github.com/pytorch/tutorials/blob/main/beginner_source/examples_autograd/polynomial_autograd.py